## Feature Scoping to determine which feature are important for the model

In [5]:
import json 
import numpy as np 
import pandas as pd 
from pathlib import Path 

ROOT = Path().cwd().parent

In [6]:
CLIENTS = ROOT / "data" / "processed" / "client"
labels = json.loads((ROOT / "data" / "processed" / "station_labels.json").read_text())

frames = {labels[p.stem]["label"]: pd.read_csv(p)
          for p in sorted(CLIENTS.glob("*.csv"))}

frames = dict(sorted(frames.items(), key=lambda kv: int(kv[0][1:])))   # S1..S7

print({k: v.shape for k, v in frames.items()})
print(list(frames["S1"].columns))


{'S1': (2470, 26), 'S2': (1627, 26), 'S3': (1789, 26), 'S4': (1769, 26), 'S5': (1770, 26), 'S6': (1759, 26), 'S7': (6799, 26)}
['station_hash_id', 'measured_ts', 'total_produced_energy', 'source', 'ac_power', 'dc_power', 'efficiency', 'device_temperature', 'perc_state_on', 'perc_state_off', 'perc_state_error', 'temperature_2m', 'shortwave_radiation', 'direct_radiation', 'diffuse_radiation', 'direct_normal_irradiance', 'global_tilted_irradiance', 'terrestrial_radiation', 'terrestrial_radiation_instant', 'global_tilted_irradiance_instant', 'direct_normal_irradiance_instant', 'diffuse_radiation_instant', 'direct_radiation_instant', 'shortwave_radiation_instant', 'tilt', 'azimuth']


In [7]:
audit = pd.DataFrame({sid: df.nunique() for sid, df in frames.items()}).T
audit

,station_hash_id,measured_ts,total_produced_energy,source,ac_power,dc_power,efficiency,device_temperature,perc_state_on,perc_state_off,...,global_tilted_irradiance,terrestrial_radiation,terrestrial_radiation_instant,global_tilted_irradiance_instant,direct_normal_irradiance_instant,diffuse_radiation_instant,direct_radiation_instant,shortwave_radiation_instant,tilt,azimuth
S1,1,2470,995,1,1458,1481,1259,1383,13,13,...,1368,1471,1224,101,12,492,38,2415,1,1
S2,1,1627,1039,1,1475,1490,1224,1398,26,26,...,1368,1471,1048,101,12,452,34,1605,1,1
S3,1,1789,1452,1,1649,1729,1579,1706,178,172,...,1508,1627,1116,101,12,453,35,1761,1,1
S4,1,1769,1192,1,1583,1575,1601,1722,204,204,...,1529,1620,637,101,12,489,25,1745,1,1
S5,1,1770,1444,1,1603,1684,1600,1727,213,213,...,1529,1620,637,101,12,490,25,1744,1,1
S6,1,1759,1328,1,1552,1671,1563,1727,207,207,...,1517,1607,637,101,12,490,25,1733,1,1
S7,1,6799,2952,1,4171,5702,5722,5877,1,1,...,3845,4151,2227,101,12,624,44,6548,1,1


## Cell 2 - Split "zero information" from "client identifier"

In [8]:
pooled = pd.concat(frames.values(), keys=frames.keys(), names=["station", "row"])

const_within = [c for c in audit.columns if (audit[c] <= 1).all()]

verdict = pd.DataFrame({
    "n_unique_pooled": pooled[const_within].nunique(),
})
verdict["verdict"] = np.where(
    verdict["n_unique_pooled"] <= 1,
    "globally constant — zero information, free drop",
    "CLIENT IDENTIFIER — varies only across clients, must drop",
)
verdict

,n_unique_pooled,verdict
station_hash_id,7,CLIENT IDENTIFIER — varies only across clients...
source,2,CLIENT IDENTIFIER — varies only across clients...
tilt,1,"globally constant — zero information, free drop"
azimuth,1,"globally constant — zero information, free drop"


## Cell 3 - State the decision rule before you lLookupError

- H1: Is terrestrial_radiation is the radation emitted by the sun?
- H2: Is it the radiation emitted by the Earth Surface?

If data shows radiation = 0 w/m^2 at night -> behaves like a solar radiation  
If the radation stays between 300 to 400 w/m^2 in day and night then it's terrestrial radiation

In [9]:
day   = pooled[pooled["global_tilted_irradiance"] >  0]
night = pooled[pooled["global_tilted_irradiance"] == 0]

for name, sub in [("day", day), ("night", night)]:
    s = sub["terrestrial_radiation"]
    print(f"{name:6s} mean={s.mean():8.1f}  max={s.max():8.1f}  frac_zero={(s == 0).mean():.3f}")

day    mean=   597.2  max=  1254.1  frac_zero=0.000
night  mean=     0.1  max=    25.1  frac_zero=0.974


## Cell 4 - Verify the radiation columns means what their names claim

In [7]:
resid = day["shortwave_radiation"] - (day["direct_radiation"] + day["diffuse_radiation"])
print(resid.abs().describe().round(3))

count    13444.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
dtype: float64


## Feature Engineering 
### Cell 5 - Two Physics Derived Feature

In [10]:
# Solar geometry: direct_horizontal = DNI × cos(zenith), so cos(zenith) is recoverable
m = day["direct_normal_irradiance"] > 10          # avoid dividing by ~0 near sunrise/sunset
cosz = day.loc[m, "direct_radiation"] / day.loc[m, "direct_normal_irradiance"]
print("cos(zenith):", cosz.describe().round(3).to_dict())   # expect bounded [0, 1]

# Clearness index: how much of the theoretical maximum actually arrived
m = day["terrestrial_radiation"] > 10
kt = day.loc[m, "shortwave_radiation"] / day.loc[m, "terrestrial_radiation"]
print("kt:", kt.describe().round(3).to_dict())              # expect ~[0, 0.8], mean ~0.4–0.6

cos(zenith): {'count': 11962.0, 'mean': 0.464, 'std': 0.25, 'min': 0.058, '25%': 0.245, '50%': 0.446, '75%': 0.669, 'max': 0.946}
kt: {'count': 13376.0, 'mean': 0.477, 'std': 0.2, 'min': 0.014, '25%': 0.318, '50%': 0.517, '75%': 0.651, 'max': 0.791}


## Cell 6 - Redundancy within and across the two column families using Pearson Correlation
- Value will be between -1 to +1
- +1 shows strong positive correlation
- -1 shows strong negative correlation

In [18]:
FAM = ["shortwave_radiation", "direct_radiation", "diffuse_radiation",
       "direct_normal_irradiance", "global_tilted_irradiance", "terrestrial_radiation"]

for c in FAM:
    print(f"{c:30s} corr(hourly_mean, instant) = {day[c].corr(day[c + '_instant']):.4f}")

print()
print(day[FAM].corr().round(2))

shortwave_radiation            corr(hourly_mean, instant) = -0.0274
direct_radiation               corr(hourly_mean, instant) = -0.1444
diffuse_radiation              corr(hourly_mean, instant) = 0.2179
direct_normal_irradiance       corr(hourly_mean, instant) = -0.3484
global_tilted_irradiance       corr(hourly_mean, instant) = -0.3349
terrestrial_radiation          corr(hourly_mean, instant) = 0.1977

                          shortwave_radiation  direct_radiation  \
shortwave_radiation                      1.00              0.96   
direct_radiation                         0.96              1.00   
diffuse_radiation                        0.67              0.45   
direct_normal_irradiance                 0.82              0.89   
global_tilted_irradiance                 0.97              0.96   
terrestrial_radiation                    0.89              0.77   

                          diffuse_radiation  direct_normal_irradiance  \
shortwave_radiation                    0.67       

In [11]:
day   = pooled[pooled["global_tilted_irradiance"] >  0]
night = pooled[pooled["global_tilted_irradiance"] == 0]

for name, sub in [("day", day), ("night", night)]:
    s = sub["terrestrial_radiation"]
    print(f"{name:6s} mean={s.mean():8.1f}  frac_zero={(s == 0).mean():.3f}")

day    mean=   597.2  frac_zero=0.000
night  mean=     0.1  frac_zero=0.974


## Resolving problematic Cos Zenith

In [12]:
import numpy as np

def add_solar_geometry(df, ts_col="measured_ts"):
    """cos_zenith and kt, both sourced from terrestrial_radiation (= TOA horizontal)."""
    ts  = pd.to_datetime(df[ts_col])
    doy = ts.dt.dayofyear

    # Earth-Sun distance correction: Earth's orbit is elliptical, so the solar
    # constant is modulated ~±3.3% over the year. Ignoring it biases cos_zenith
    # seasonally — which would alias straight into your doy_sin/doy_cos features.
    E0 = 1 + 0.033 * np.cos(2 * np.pi * doy / 365.0)

    # Invert TOA = I_sc * E0 * cos(z). Defined EVERYWHERE, including night (-> 0)
    # and sunrise/sunset, unlike the DNI-ratio version. No weather input.
    df["cos_zenith"] = (df["terrestrial_radiation"] / (1361.0 * E0)).clip(0, 1)

    # Clearness index: fraction of the theoretical ceiling that survived the
    # atmosphere. This is the ONLY weather signal in the pair - it's what
    # separates "sun is low" from "sun is behind a cloud".
    denom = df["terrestrial_radiation"]
    df["kt"] = np.where(denom > 10, df["shortwave_radiation"] / denom, np.nan)

    df["is_daylight"] = (denom > 10).astype("int8")
    return df

In [13]:
chk = add_solar_geometry(day.copy())
m = day["direct_normal_irradiance"] > 10
old = day.loc[m, "direct_radiation"] / day.loc[m, "direct_normal_irradiance"]
new = chk.loc[m, "cos_zenith"]

print(f"corr        : {old.corr(new):.4f}")          # expect > 0.99
print(f"max abs dev : {(old - new).abs().max():.4f}")  # expect small
print(f"coverage    : {old.notna().sum()} -> {chk['cos_zenith'].notna().sum()}")

corr        : 0.9991
max abs dev : 0.0442
coverage    : 11962 -> 13444


In [14]:
FEATURE_SPEC = {
    "target": "capacity_factor",                     # D-010

    # --- Layer 1: what the builder must READ. Not model inputs. ---
    "source_columns": [
        "measured_ts", "total_produced_energy",
        "temperature_2m", "shortwave_radiation", "terrestrial_radiation",
        "direct_radiation", "diffuse_radiation", "global_tilted_irradiance",
    ],

    # --- Layer 2: derived, with explicit dependencies ---
    "derived": {
        "cos_zenith":  {"from": ["terrestrial_radiation", "measured_ts"],
                        "rule": "TOA / (1361 * E0(doy)), clipped [0,1]"},
        "kt":          {"from": ["shortwave_radiation", "terrestrial_radiation"],
                        "rule": "GHI / TOA", "guard": "TOA > 10 W/m2 else NaN"},
        "is_daylight": {"from": ["terrestrial_radiation"], "rule": "TOA > 10"},
        "hour_sin": {"from": ["measured_ts"]}, "hour_cos": {"from": ["measured_ts"]},
        "doy_sin":  {"from": ["measured_ts"]}, "doy_cos":  {"from": ["measured_ts"]},
    },

    # --- Layer 3: the model matrix ---
    "model_matrix": {
        "history":  ["capacity_factor"],
        "geometry": ["cos_zenith", "hour_sin", "hour_cos",
                     "doy_sin", "doy_cos", "is_daylight"],
        "weather_past":   ["temperature_2m", "direct_radiation",
                           "diffuse_radiation", "global_tilted_irradiance", "kt"],
        "weather_future": ["temperature_2m", "direct_radiation",
                           "diffuse_radiation", "global_tilted_irradiance", "kt"],
    },

    # --- Layer 4: exclusions, each with the reason it was excluded ---
    "excluded": {
        "consumed_by_derived": {
            "shortwave_radiation":   "numerator of kt; also == direct + diffuse exactly",
            "terrestrial_radiation": "source of cos_zenith and kt denominator; "
                                     "pure geometry, carries no weather signal",
        },
        "leaky_client_id":   ["station_hash_id", "source"],
        "globally_constant": ["tilt", "azimuth"],
        "unreliable":        ["*_instant"],   # see note below - NOT redundant
    },
}

(ROOT / "configs" / "features.yaml").write_text(
    __import__("yaml").safe_dump(FEATURE_SPEC, sort_keys=False))

1401